In [1]:
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np
from wrf import getvar, ALL_TIMES
from metpy.calc import wet_bulb_temperature, dewpoint_from_specific_humidity
from metpy.units import units
import matplotlib.animation as animation
from IPython.display import HTML
import os
import glob

# read WRF output and subset region of interest

In [2]:
def process_member(files, member_name):
    times = []
    wind_list = []
    wetbulb_list = []
    precip_list = []
    sr_list = []

    prev_total = None

    for i in range(len(files)):
        ds = xr.open_dataset(files[i])
        
        # TIME
        time_str = "".join(ds.Times.values.astype(str))
        valid_time = np.datetime64(time_str.replace("_", " "))
        times.append(valid_time)
        
        # WIND
        wind = np.sqrt(ds.U10**2 + ds.V10**2)
        wind_list.append(wind.squeeze())
        
        # WET BULB
        pressure = (ds.PSFC / 100.0) * units.hPa
        temperature = (ds.T2 - 273.15) * units.degC
        q = ds.Q2 * units.dimensionless
        dewpoint = dewpoint_from_specific_humidity(pressure, temperature, q)
        wetbulb = wet_bulb_temperature(pressure, temperature, dewpoint).metpy.convert_units("degC")
        wetbulb_list.append(wetbulb.squeeze())
        
        # PRECIP RATE (optimized)
        total_now = ds.RAINC + ds.RAINNC
        
        if prev_total is not None:
            rate = (total_now - prev_total) / 0.5
        else:
            rate = xr.zeros_like(total_now)
        
        precip_list.append(rate.squeeze())
        prev_total = total_now
        
        # SR
        if "SR" in ds.variables:
            sr_list.append(ds.SR.squeeze())
        else:
            sr_proxy = (ds.QICE + ds.QSNOW + ds.QGRAUP) / (
                ds.QICE + ds.QSNOW + ds.QGRAUP + ds.QRAIN + ds.QCLOUD
            )
            sr_list.append(sr_proxy.squeeze())
        
        ds.close()

    wind_all = xr.concat(wind_list, dim="time").assign_coords(time=times)
    wetbulb_all = xr.concat(wetbulb_list, dim="time").assign_coords(time=times)
    precip_all = xr.concat(precip_list, dim="time").assign_coords(time=times)
    sr_all = xr.concat(sr_list, dim="time").assign_coords(time=times)

    # Drop XTIME cleanly
    for da in [wind_all, wetbulb_all, precip_all, sr_all]:
        if "XTIME" in da.coords:
            da = da.drop_vars("XTIME")

    wind_all = wind_all.drop_vars("XTIME", errors="ignore")
    wetbulb_all = wetbulb_all.drop_vars("XTIME", errors="ignore")
    precip_all = precip_all.drop_vars("XTIME", errors="ignore")
    sr_all = sr_all.drop_vars("XTIME", errors="ignore")

    ds_full = xr.Dataset({
        "wind_speed": wind_all,
        "wetbulb": wetbulb_all,
        "precip_rate": precip_all,
        "SR": sr_all
    })

    ds_clean = ds_full[["precip_rate", "SR", "wind_speed", "wetbulb"]]

    if "Time" in ds_clean.dims:
        ds_clean = ds_clean.rename({"Time": "time"})

    # Add ensemble label HERE
    ds_clean = ds_clean.expand_dims({"ensemble": [member_name]})

    return ds_clean

In [3]:
base_path = "/data/scratch/a/kevintg2/I-HRRRens"
members = sorted(os.listdir(base_path))

all_members = []

completed = {
    f for f in os.listdir("/data/keeling/a/alison8/AlisonCapstone")
    if f.startswith("wrf_framinputs_") and f.endswith(".nc")
}

for m in members:
    outfile = f"wrf_framinputs_{m}_14dec2024.nc"
    if outfile in completed:
        continue  # skip, already done
    
    print(f"Processing member: {m}")
    path = f"{base_path}/{m}/output/yesterday/archive_20241214/wrfout*"
    files = sorted(glob.glob(path))
    
    ds_member = process_member(files, m)
    
    # Save individual file
    out_path = f"/data/keeling/a/alison8/AlisonCapstone/wrf_framinputs_{m}_14dec2024.nc"
    
    encoding = {var: {"zlib": True, "complevel": 4} for var in ds_member.data_vars}
    ds_member.to_netcdf(out_path, encoding=encoding)
    print(f"Saved: {out_path}")

Processing member: Thompson_MYNN
Saved: /data/keeling/a/alison8/AlisonCapstone/wrf_framinputs_Thompson_MYNN_14dec2024.nc
Processing member: Thompson_YSU
Saved: /data/keeling/a/alison8/AlisonCapstone/wrf_framinputs_Thompson_YSU_14dec2024.nc
Processing member: WDM7_MYJ


/data/keeling/a/alison8/miniconda3/envs/ATMS596/lib/python3.13/site-packages/metpy/calc/thermo.py:1753: RuntimeWarning: invalid value encountered in log
  val = np.log(vapor_pressure / mpconsts.nounit.sat_pressure_0c)


Saved: /data/keeling/a/alison8/AlisonCapstone/wrf_framinputs_WDM7_MYJ_14dec2024.nc
Processing member: WDM7_MYNN
Saved: /data/keeling/a/alison8/AlisonCapstone/wrf_framinputs_WDM7_MYNN_14dec2024.nc
Processing member: WDM7_YSU
Saved: /data/keeling/a/alison8/AlisonCapstone/wrf_framinputs_WDM7_YSU_14dec2024.nc
Processing member: run_scripts


ValueError: must supply at least one object to concatenate

# compute ice accretion

# stack ensemble dimension

# plot all ensemble members on one figure

# ensemble statistics